In [1]:
import subprocess
subprocess.run(["pip", "install", "marker-pdf", "Pillow", "tqdm", "--quiet"], check=True)
print("Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.109.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


Done


In [4]:
import os, re, json, hashlib, shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm

PDF_DIR       = Path("/kaggle/input/datasets/vudathab/sample-data-knowledge-assistant/upload_to_kaggle/pdfs/")
OUTPUT_DIR    = Path("/kaggle/working")
IMAGES_DIR    = OUTPUT_DIR / "images"
METADATA_FILE = OUTPUT_DIR / "metadata.jsonl"
LOG_FILE      = OUTPUT_DIR / "extraction_log.jsonl"

MAX_CHUNK_WORDS   = 400
MIN_SECTION_WORDS = 80


SKIP_SECTIONS = {
    "references", "bibliography", "acknowledgements",
    "acknowledgments", "appendix", "funding",
    "conflicts of interest", "author contributions",
    "supplementary", "ethics statement"
}

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# ── load metadata from YOUR sample log ───────────────────────
paper_meta = {}
log_path = Path("/kaggle/input/datasets/vudathab/sample-data-knowledge-assistant/upload_to_kaggle/download_log.jsonl")
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("status") == "success":
                paper_meta[rec["arxiv_id"]] = {
                    "title"   : rec["title"],
                    "category": rec["category"],
                }
    print(f"Metadata loaded : {len(paper_meta)} papers")
else:
    print("WARNING: download_log.jsonl not found — titles will fall back to arxiv_id")

# ── load ALL pdfs in sample folder, no cap ───────────────────
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print(f"PDFs to process : {len(pdf_files)}")
for p in pdf_files:
    arxiv_id = p.stem.replace("_", ".")
    meta     = paper_meta.get(arxiv_id, {})
    title    = meta.get("title", "NOT IN LOG")
    category = meta.get("category", "???")
    print(f"  {p.name:<20}  [{category}]  {title[:50]}")

print(f"\nCUDA available  : {__import__('torch').cuda.is_available()}")

Metadata loaded : 21 papers
PDFs to process : 21
  1706.03762.pdf        [Transformers & Attention]  Attention Is All You Need
  2001.04451.pdf        [Transformers & Attention]  Reformer: The Efficient Transformer
  2005.11401.pdf        [RAG & Retrieval]  Retrieval-Augmented Generation for Knowledge-Inten
  2005.14165.pdf        [LLMs & Scaling]  Language Models are Few-Shot Learners (GPT-3)
  2101.0019.pdf         [Efficient & Fine-tuning]  Prefix-Tuning: Optimizing Continuous Prompts for G
  2103.0002.pdf         [CV & Multimodal]  Learning Transferable Visual Models from Natural L
  2209.1443.pdf         [Diffusion & Generation]  Imagic: Text-Based Real Image Editing with Diffusi
  2304.01196.pdf        [Efficient & Fine-tuning]  Sparks of Artificial General Intelligence: Early e
  2305.14283.pdf        [RAG & Retrieval]  FLARE: Active Retrieval Augmented Generation
  2305.16103.pdf        [Diffusion & Generation]  Drag Your GAN: Interactive Point-based Manipulatio
  2305.16291.pd

In [5]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

print("[+] Loading Marker models (~2GB first run)...")
model_dict = create_model_dict()
converter  = PdfConverter(artifact_dict=model_dict)
print("[+] Done — models ready")

2026-05-04 00:10:27.101459: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777853427.314376      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777853427.372052      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777853427.875728      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777853427.875776      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777853427.875778      57 computation_placer.cc:177] computation placer alr

[+] Loading Marker models (~2GB first run)...


[+] Done — models ready


In [15]:
# ── HTML + text cleaning ─────────────────────────────────────

def strip_author_block(text: str) -> str:
    """
    Remove author name/email blocks that Marker puts at the top
    of the first section. Pattern: lines with email addresses.
    """
    lines = text.split('\n')
    clean_lines = []
    skip_zone = False

    for line in lines:
        # detect email line — signals start of author block
        if re.search(r'[\w.\-]+@[\w.\-]+\.\w+', line):
            skip_zone = True
            continue
        # author blocks end at a blank line followed by real content
        if skip_zone and line.strip() == '':
            continue
        if skip_zone and len(line.strip()) > 60:
            skip_zone = False
        if not skip_zone:
            clean_lines.append(line)

    return '\n'.join(clean_lines)


def clean_section_name(heading: str) -> str:
    heading = re.sub(r'^#+\s*', '', heading)
    heading = strip_html(heading)
    heading = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', heading)
    return heading.strip()


def clean_text(text: str) -> str:
    # 1. stash LaTeX so cleaning never touches it
    equations = {}
    def stash(m):
        key = f"__EQ{len(equations)}__"
        equations[key] = m.group()
        return key
    text = re.sub(r'\$\$.*?\$\$', stash, text, flags=re.DOTALL)
    text = re.sub(r'\$[^$\n]+\$', stash, text)

    # 2. strip HTML
    text = strip_html(text)

    # 3. FIX: remove escaped markdown links [\[13\]](#page-10-0) style
    text = re.sub(r'\[\\*\[[\d,\-\s]+\\*\]\]\([^\)]*\)', '', text)

    # 4. remove standard markdown links → keep visible text
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)

    # 5. bare URLs
    text = re.sub(r'https?://\S+', '', text)
    # 6. citation markers [1], [1,2], [1-4]
    text = re.sub(r'\[\d+(?:[,\-]\s*\d+)*\]', '', text)

    # 7. (Author et al., 2017) style
    text = re.sub(r'\([A-Z][a-z]+(?:\s+et\s+al\.?)?,?\s*\d{4}\)', '', text)

    # 8. isolated page numbers
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)

    # 9. whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 10. restore equations
    for key, eq in equations.items():
        text = text.replace(key, eq)

    return text.strip()


# ── Caption extraction ────────────────────────────────────────

def extract_captions(markdown: str) -> dict:
    caption_map = {}

    # Format A: ![caption here](key)
    for m in re.finditer(r'!\[([^\]]+)\]\(([^\)]+)\)', markdown):
        caption, key = m.group(1).strip(), m.group(2).strip()
        if caption:
            caption_map[key] = caption

    # Format B/C: ![](key) then "Figure N: ..." on next 1-2 lines
    for m in re.finditer(
        r'!\[\]\(([^\)]+)\)\n{1,2}(Figure[^\n]+)', markdown
    ):
        key = m.group(1).strip()
        if key not in caption_map:
            caption_map[key] = m.group(2).strip()

    # Format D: ![](key) then "Fig. N ..." variant
    for m in re.finditer(
        r'!\[\]\(([^\)]+)\)\n{1,2}(Fig\.?\s*\d+[^\n]+)', markdown
    ):
        key = m.group(1).strip()
        if key not in caption_map:
            caption_map[key] = m.group(2).strip()

    # Format E: FIX for Figure 1 — caption appears up to 5 lines after image
    # Marker sometimes inserts blank lines or small text between image and caption
    for m in re.finditer(
        r'!\[\]\(([^\)]+)\)((?:\n[^\n]*){0,5}?\n)(Figure\s*\d+[^\n]+)',
        markdown
    ):
        key     = m.group(1).strip()
        caption = m.group(3).strip()
        if key not in caption_map and caption:
            caption_map[key] = caption

    # Format F: scan entire markdown for "Figure N: ..." lines
    # then backtrack to find the nearest image key above it
    figure_line_pattern = re.compile(
        r'(Figure\s*\d+\s*[:\.][^\n]+)', re.IGNORECASE
    )
    image_ref_pattern = re.compile(r'!\[\]\(([^\)]+)\)')

    # find all image positions and caption positions
    image_positions = [
        (m.start(), m.group(1)) for m in image_ref_pattern.finditer(markdown)
    ]
    for cap_match in figure_line_pattern.finditer(markdown):
        caption    = cap_match.group(1).strip()
        cap_start  = cap_match.start()

        # find the nearest image reference BEFORE this caption
        # within a 500 character window
        nearest_key = None
        nearest_dist = 999
        for img_pos, img_key in image_positions:
            dist = cap_start - img_pos
            if 0 < dist < 500 and dist < nearest_dist:
                nearest_dist = dist
                nearest_key  = img_key

        if nearest_key and nearest_key not in caption_map:
            caption_map[nearest_key] = caption

    return caption_map


# ── Chunking ──────────────────────────────────────────────────

def count_words(text: str) -> int:
    return len(text.split())


def is_skip_section(heading: str) -> bool:
    return heading.lower().strip().rstrip('.') in SKIP_SECTIONS


def split_at_sentences(text: str) -> list:
    # protect LaTeX and abbreviations from being split
    text = re.sub(r'\$\$.*?\$\$', lambda m: m.group().replace('.', '<DOT>'),
                  text, flags=re.DOTALL)
    text = re.sub(r'\$[^$]+\$',   lambda m: m.group().replace('.', '<DOT>'), text)
    text = re.sub(r'\b(Fig|Tab|Eq|Sec|et al|vs|approx|e\.g|i\.e)\.',
                  lambda m: m.group().replace('.', '<DOT>'), text)
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [p.replace('<DOT>', '.').strip() for p in parts if p.strip()]


def pack_sentences(sentences: list, max_words: int) -> list:
    chunks, current, current_words = [], [], 0
    for sent in sentences:
        w = count_words(sent)
        if current_words + w <= max_words:
            current.append(sent)
            current_words += w
        else:
            if current:
                chunks.append(' '.join(current))
            current, current_words = [sent], w
    if current:
        chunks.append(' '.join(current))
    return chunks


def smart_chunk(text: str, max_words: int = MAX_CHUNK_WORDS) -> list:
    text = text.strip()
    if not text:
        return []
    if count_words(text) <= max_words:
        return [text]

    paragraphs = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
    chunks, current_paras, current_words = [], [], 0

    for para in paragraphs:
        para_words = count_words(para)
        if current_words + para_words <= max_words:
            current_paras.append(para)
            current_words += para_words
        elif para_words > max_words:
            if current_paras:
                chunks.append('\n\n'.join(current_paras))
                current_paras, current_words = [], 0
            sentences = split_at_sentences(para)
            chunks.extend(pack_sentences(sentences, max_words))
        else:
            if current_paras:
                chunks.append('\n\n'.join(current_paras))
            current_paras, current_words = [para], para_words

    if current_paras:
        chunks.append('\n\n'.join(current_paras))

    return [c for c in chunks if c.strip()]


def chunk_with_tables(text: str, max_words: int = MAX_CHUNK_WORDS) -> list:
    """
    Extract markdown tables as atomic chunks first,
    then smart_chunk everything else. Tables never get split mid-row.
    """
    table_pattern = re.compile(
        r'(\|.+\|\n\|[-| :]+\|\n(?:\|.+\|\n?)+)',
        re.MULTILINE
    )
    chunks, last_end = [], 0
    for match in table_pattern.finditer(text):
        pre = text[last_end:match.start()].strip()
        if pre:
            chunks.extend(smart_chunk(pre, max_words))
        chunks.append(match.group().strip())   # table stays whole
        last_end = match.end()
    tail = text[last_end:].strip()
    if tail:
        chunks.extend(smart_chunk(tail, max_words))
    return chunks


def build_context_header(title, arxiv_id, section, idx, total):
    return (
        f"Paper: {title} [{arxiv_id}]\n"
        f"Section: {section}\n"
        f"Chunk: {idx + 1} of {total}\n\n"
    )


print("[+] Utilities loaded")

[+] Utilities loaded


In [16]:
def parse_marker_output(markdown, images, arxiv_id, paper_title, category):

    text_records   = []
    figure_records = []
    chunk_counter  = 0

    # ── build caption map (all 4 formats) ────────────────────
    caption_map = extract_captions(markdown)

    # ── parse sections ────────────────────────────────────────
    sections     = {}
    section_order = []
    current_section = "Abstract"
    current_content = []
    skip_current    = False

    for line in markdown.split('\n'):
        heading_match = re.match(r'^#{1,3}\s+(.+)$', line)

        if heading_match:
            # flush previous section
            if current_content and not skip_current:
                section_text = '\n'.join(current_content).strip()
                if section_text:
                    sections.setdefault(current_section, []).append(section_text)
                    if current_section not in section_order:
                        section_order.append(current_section)

            # clean the heading — strips HTML from section names
            current_section = clean_section_name(heading_match.group(1))
            current_content = []
            skip_current    = is_skip_section(current_section)
        else:
            if not skip_current:
                current_content.append(line)

    # flush last section
    if current_content and not skip_current:
        section_text = '\n'.join(current_content).strip()
        if section_text:
            sections.setdefault(current_section, []).append(section_text)
            if current_section not in section_order:
                section_order.append(current_section)

    # ── chunk text sections ───────────────────────────────────
    pending_merge = None

    for section_name in section_order:
        content_list = sections[section_name]
        raw_text     = '\n\n'.join(content_list)

        # strip image refs from text (figures handled separately)
        raw_text = re.sub(r'!\[([^\]]*)\]\([^\)]+\)', '', raw_text)

        full_text = clean_text(raw_text)
        # ADD THIS in Cell 5 parse_marker_output, after full_text = clean_text(raw_text)
        if section_name in ("Abstract", list(sections.keys())[0]):
            full_text = strip_author_block(full_text)

        if not full_text:
            continue

        word_count = count_words(full_text)

        # merge tiny sections into next section
        if word_count < MIN_SECTION_WORDS:
            if pending_merge is None:
                pending_merge = (section_name, full_text)
            else:
                prev_name, prev_text = pending_merge
                pending_merge = (
                    f"{prev_name} / {section_name}",
                    prev_text + "\n\n" + full_text
                )
            continue

        # flush any pending merge into this section
        if pending_merge is not None:
            prev_name, prev_text = pending_merge
            full_text    = prev_text + "\n\n" + full_text
            section_name = f"{prev_name} / {section_name}"
            pending_merge = None

        # chunk — tables stay whole, text splits at sentences
        raw_chunks = chunk_with_tables(full_text, MAX_CHUNK_WORDS)
        total      = len(raw_chunks)

        for i, raw_chunk in enumerate(raw_chunks):
            header       = build_context_header(paper_title, arxiv_id, section_name, i, total)
            context_text = header + raw_chunk

            text_records.append({
                "id"          : f"{arxiv_id}_text_{chunk_counter:04d}",
                "record_type" : "text",
                "source"      : "arxiv",
                "arxiv_id"    : arxiv_id,
                "paper_title" : paper_title,
                "category"    : category,
                "section"     : section_name,
                "chunk_index" : i,
                "chunk_total" : total,
                "page"        : 0,
                "word_count"  : count_words(raw_chunk),
                "text"        : raw_chunk,
                "context_text": context_text,
                "image_path"  : None,
            })
            chunk_counter += 1

    # flush final pending merge
    if pending_merge is not None:
        section_name, text = pending_merge
        if count_words(text) > 20:
            header = build_context_header(paper_title, arxiv_id, section_name, 0, 1)
            text_records.append({
                "id"          : f"{arxiv_id}_text_{chunk_counter:04d}",
                "record_type" : "text",
                "source"      : "arxiv",
                "arxiv_id"    : arxiv_id,
                "paper_title" : paper_title,
                "category"    : category,
                "section"     : section_name,
                "chunk_index" : 0,
                "chunk_total" : 1,
                "page"        : 0,
                "word_count"  : count_words(text),
                "text"        : text,
                "context_text": header + text,
                "image_path"  : None,
            })

    # ── extract figures ───────────────────────────────────────
    for image_key, pil_image in images.items():
        if pil_image is None:
            continue
        if pil_image.width < 100 or pil_image.height < 100:
            continue

        img_hash = hashlib.md5(pil_image.tobytes()).hexdigest()[:12]
        img_name = f"{arxiv_id}_{img_hash}.png"
        img_path = IMAGES_DIR / img_name

        try:
            if pil_image.mode != "RGB":
                pil_image = pil_image.convert("RGB")
            pil_image.save(img_path, "PNG")
        except Exception:
            continue

        caption = caption_map.get(image_key, "")
        if not caption:
            caption = f"Figure from {paper_title}"

        fig_match = re.search(r'(?:Figure|Fig\.?)\s*(\d+)', caption, re.I)
        fig_num   = fig_match.group(1) if fig_match else str(len(figure_records) + 1)

        context_text = (
            f"Paper: {paper_title} [{arxiv_id}]\n"
            f"Section: Figure\n\n{caption}"
        )

        figure_records.append({
            "id"          : f"{arxiv_id}_fig_{img_hash}",
            "record_type" : "figure",
            "source"      : "arxiv",
            "arxiv_id"    : arxiv_id,
            "paper_title" : paper_title,
            "category"    : category,
            "section"     : "Figure",
            "chunk_index" : 0,
            "chunk_total" : 1,
            "page"        : 0,
            "word_count"  : count_words(caption),
            "text"        : context_text,
            "context_text": context_text,
            "image_path"  : str(img_path),
            "caption"     : caption,
            "figure_num"  : fig_num,
        })

    return text_records, figure_records


print("[+] Parser loaded")

[+] Parser loaded


In [17]:
stats = {"processed": 0, "failed": 0, "text": 0, "figures": 0, "words": 0}

jsonl_handle = open(METADATA_FILE, "w", encoding="utf-8")
log_handle   = open(LOG_FILE,      "w", encoding="utf-8")

def write_record(rec):
    jsonl_handle.write(json.dumps(rec) + "\n")

def write_log(arxiv_id, status, text_count, fig_count, error=""):
    log_handle.write(json.dumps({
        "arxiv_id": arxiv_id, "status": status,
        "text_chunks": text_count, "figures": fig_count, "error": error,
    }) + "\n")

print(f"[+] Processing {len(pdf_files)} PDFs...\n")

for pdf_path in tqdm(pdf_files, desc="Extracting"):
    arxiv_id    = pdf_path.stem.replace("_", ".")
    meta        = paper_meta.get(arxiv_id, {})
    paper_title = meta.get("title", arxiv_id)
    category    = meta.get("category", "Unknown")

    try:
        rendered = converter(str(pdf_path))

        # Marker v2 API — fall back to v1 if needed
        try:
            markdown = rendered.markdown
            images   = rendered.images
        except AttributeError:
            from marker.output import text_from_rendered
            markdown, _, images = text_from_rendered(rendered)

        if not markdown or len(markdown.strip()) < 100:
            write_log(arxiv_id, "failed", 0, 0, "empty output")
            stats["failed"] += 1
            continue

        text_recs, fig_recs = parse_marker_output(
            markdown, images, arxiv_id, paper_title, category
        )

        if not text_recs:
            write_log(arxiv_id, "failed", 0, 0, "no text records")
            stats["failed"] += 1
            continue

        for rec in text_recs:
            write_record(rec)
        for rec in fig_recs:
            write_record(rec)

        stats["processed"] += 1
        stats["text"]      += len(text_recs)
        stats["figures"]   += len(fig_recs)
        stats["words"]     += sum(r["word_count"] for r in text_recs)

        write_log(arxiv_id, "success", len(text_recs), len(fig_recs))
        tqdm.write(f"  OK  {arxiv_id} — {len(text_recs)} chunks, {len(fig_recs)} figs")

    except Exception as e:
        tqdm.write(f"  FAIL {arxiv_id}: {e}")
        write_log(arxiv_id, "failed", 0, 0, str(e))
        stats["failed"] += 1

jsonl_handle.close()
log_handle.close()

avg_wds = stats["words"] // stats["text"] if stats["text"] else 0
total   = stats["text"] + stats["figures"]
print(f"""
[+] Done!
    Processed  : {stats['processed']} PDFs
    Failed     : {stats['failed']} PDFs
    Text chunks: {stats['text']:,}
    Figures    : {stats['figures']:,}
    Total      : {total:,}
    Avg words  : {avg_wds}
""")

[+] Processing 21 PDFs...



Extracting:   0%|          | 0/21 [00:00<?, ?it/s]

Recognizing Layout:   0%|          | 0/15 [00:00<?, ?it/s]

Recognizing Layout:   7%|▋         | 1/15 [00:07<01:42,  7.34s/it]

Recognizing Layout:  20%|██        | 3/15 [00:07<00:23,  1.95s/it]

Recognizing Layout:  27%|██▋       | 4/15 [00:09<00:23,  2.15s/it]

Recognizing Layout:  67%|██████▋   | 10/15 [00:10<00:02,  1.80it/s]

Recognizing Layout: 100%|██████████| 15/15 [00:10<00:00,  1.46it/s]


Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 31.60it/s]


Detecting bboxes:   0%|          | 0/1 [00:00<?, ?it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  1.00it/s]


Recognizing Text:   0%|          | 0/40 [00:00<?, ?it/s]

Recognizing Text:   2%|▎         | 1/40 [00:02<01:53,  2.90s/it]

Recognizing Text:  15%|█▌        | 6/40 [00:03<00:17,  1.95it/s]

Recognizing Text:  18%|█▊        | 7/40 [00:04<00:20,  1.59it/s]

Recognizing Text:  20%|██        | 8/40 [00:05<00:17,  1.88it/s]

Recognizing Text:  28%|

  OK  1706.03762 — 27 chunks, 6 figs




Recognizing Layout:   0%|          | 0/12 [00:00<?, ?it/s]

Recognizing Layout:   8%|▊         | 1/12 [00:07<01:18,  7.11s/it]

Recognizing Layout:  17%|█▋        | 2/12 [00:07<00:29,  2.99s/it]

Recognizing Layout: 100%|██████████| 12/12 [00:07<00:00,  1.62it/s][A


Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00, 23.68it/s]


Detecting bboxes:   0%|          | 0/1 [00:00<?, ?it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s]


Recognizing Text:   0%|          | 0/42 [00:00<?, ?it/s]

Recognizing Text:   2%|▏         | 1/42 [00:04<02:58,  4.35s/it]

Recognizing Text:  10%|▉         | 4/42 [00:04<00:32,  1.17it/s]

Recognizing Text:  14%|█▍        | 6/42 [00:07<00:39,  1.09s/it]

Recognizing Text:  17%|█▋        | 7/42 [00:08<00:37,  1.08s/it]

Recognizing Text:  19%|█▉        | 8/42 [00:08<00:32,  1.03it/s]

Recognizing Text:  29%|██▊       | 12/42 [00:12<00:28,  1.07it/s]

Recognizing Text:  31%|███       | 13/42 [00:13<00:28,  1.01it/s]

Recogniz

  OK  2001.04451 — 24 chunks, 5 figs


...

  OK  2403.12015 — 25 chunks, 28 figs




Recognizing Layout:   0%|          | 0/36 [00:00<?, ?it/s]

Recognizing Layout:   3%|▎         | 1/36 [00:06<03:55,  6.74s/it]

Recognizing Layout:  11%|█         | 4/36 [00:08<00:55,  1.73s/it]

Recognizing Layout:  17%|█▋        | 6/36 [00:08<00:29,  1.00it/s]

Recognizing Layout:  22%|██▏       | 8/36 [00:10<00:29,  1.05s/it]

Recognizing Layout:  33%|███▎      | 12/36 [00:13<00:19,  1.26it/s]

Recognizing Layout:  42%|████▏     | 15/36 [00:14<00:15,  1.38it/s]

Recognizing Layout:  53%|█████▎    | 19/36 [00:17<00:11,  1.47it/s]

Recognizing Layout:  56%|█████▌    | 20/36 [00:17<00:09,  1.65it/s]

Recognizing Layout:  61%|██████    | 22/36 [00:19<00:09,  1.45it/s]

Recognizing Layout:  67%|██████▋   | 24/36 [00:19<00:06,  1.93it/s]

Recognizing Layout:  69%|██████▉   | 25/36 [00:21<00:07,  1.39it/s]

Recognizing Layout:  78%|███████▊  | 28/36 [00:21<00:03,  2.30it/s]

Recognizing Layout:  86%|████████▌ | 31/36 [00:21<00:01,  3.35it/s]

Recognizing Layout:  94%|█████████▍| 34/36 [0

  OK  2404.03715 — 79 chunks, 2 figs




Recognizing Layout:   0%|          | 0/30 [00:00<?, ?it/s]

Recognizing Layout:   3%|▎         | 1/30 [00:06<03:15,  6.75s/it]

Recognizing Layout:  13%|█▎        | 4/30 [00:08<00:45,  1.74s/it]

Recognizing Layout:  23%|██▎       | 7/30 [00:10<00:25,  1.12s/it]

Recognizing Layout:  33%|███▎      | 10/30 [00:11<00:17,  1.14it/s]

Recognizing Layout:  40%|████      | 12/30 [00:11<00:11,  1.58it/s]

Recognizing Layout:  43%|████▎     | 13/30 [00:13<00:14,  1.21it/s]

Recognizing Layout:  57%|█████▋    | 17/30 [00:16<00:09,  1.38it/s]

Recognizing Layout:  67%|██████▋   | 20/30 [00:17<00:06,  1.62it/s]

Recognizing Layout:  83%|████████▎ | 25/30 [00:17<00:01,  2.87it/s]

Recognizing Layout:  90%|█████████ | 27/30 [00:17<00:00,  3.37it/s]

Recognizing Layout: 100%|██████████| 30/30 [00:18<00:00,  1.64it/s]


Running OCR Error Detection:   0%|          | 0/3 [00:00<?, ?it/s]

Running OCR Error Detection: 100%|██████████| 3/3 [00:00<00:00, 19.73it/s]


Detecting bboxes:   0%|          | 0

  OK  2404.04292 — 48 chunks, 4 figs

[+] Done!
    Processed  : 21 PDFs
    Failed     : 0 PDFs
    Text chunks: 998
    Figures    : 226
    Total      : 1,224
    Avg words  : 225



In [19]:
with open(METADATA_FILE, encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

text_recs = [r for r in records if r["record_type"] == "text"]
fig_recs  = [r for r in records if r["record_type"] == "figure"]

print("=" * 60)
print("SECTION NAMES (first 15)")
print("=" * 60)
seen = set()
for r in text_recs:
    s = r["section"]
    if s not in seen:
        print(repr(s))
        seen.add(s)
    if len(seen) >= 15:
        break

print("\n" + "=" * 60)
print("SAMPLE TEXT CHUNKS (first 3, raw)")
print("=" * 60)
for r in text_recs[:3]:
    print(f"\n[{r['arxiv_id']}] section={r['section']!r}")
    print(repr(r["text"][:500]))

print("\n" + "=" * 60)
print("FIGURE CAPTIONS (first 5)")
print("=" * 60)
for r in fig_recs[:5]:
    print(f"\n[{r['arxiv_id']}] fig_num={r['figure_num']}")
    print(f"  caption : {r['caption']}")
    print(f"  img     : {Path(r['image_path']).name}")

print("\n" + "=" * 60)
print("EQUATION CHECK")
print("=" * 60)
eq_chunks = [r for r in text_recs if "$" in r["text"]]
print(f"Chunks with LaTeX: {len(eq_chunks)}")
if eq_chunks:
    print(repr(eq_chunks[0]["text"][:400]))

print("\n" + "=" * 60)
print("HTML NOISE CHECK")
print("=" * 60)
html_chunks = [r for r in text_recs if re.search(r'<[a-z]+', r["text"])]
print(f"Chunks still containing HTML tags: {len(html_chunks)}")
if html_chunks:
    for r in html_chunks[:2]:
        print(repr(r["text"][:300]))

SECTION NAMES (first 15)
'Abstract'
'Attention Is All You Need / 1 Introduction'
'2 Background'
'3 Model Architecture'
'3.1 Encoder and Decoder Stacks'
'3.2 Attention'
'3.2.3 Applications of Attention in our Model'
'3.3 Position-wise Feed-Forward Networks'
'3.4 Embeddings and Softmax'
'4 Why Self-Attention'
'5 Training / 5.1 Training Data and Batching'
'5.2 Hardware and Schedule / 5.3 Optimizer / 5.4 Regularization'
'6.3 English Constituency Parsing'
'7 Conclusion'
'Attention Visualizations **Input-Input Layer5**'

SAMPLE TEXT CHUNKS (first 3, raw)

[1706.03762] section='Abstract'
'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mech

In [20]:
shutil.make_archive(str(OUTPUT_DIR / "extracted"), "zip", str(OUTPUT_DIR), ".")
size_mb = os.path.getsize(str(OUTPUT_DIR / "extracted.zip")) / 1024**2
print(f"extracted.zip ready — {size_mb:.1f} MB")
print("Download from Kaggle Output tab")

extracted.zip ready — 34.2 MB
Download from Kaggle Output tab
